# ApexPINN: diagnose saved M1 models

Prepared, NOT RUN. Select a hosted L4 GPU. This notebook performs inference on saved models only.
It does not train, tune, run a restart oracle, change checkpoints or replace forecasts.

Post-result method `m1_offset_diagnostic/v1`, declared 2026-11-05: inspect both calibration and final
M1/F1/mid/main fits. Compare trained offsets with the least-squares offsets on the same training
labels; split residual error into offset and shape components. Report all five members and plot
fixed batch rows. No decision threshold or pivot is introduced. Profiled training errors are not
held-out scores. Preserve the completed unfavorable result and use a separate output directory.


In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive, files
except ImportError as exc:
    raise RuntimeError('Use a hosted Colab GPU. Local execution is prohibited.') from exc
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('A hosted Colab runtime is required.')
drive.mount('/content/drive')
os.environ['APEXPINN_REMOTE_RUNTIME'] = 'colab'


## 2. Select saved inputs and a new diagnostic ID


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/apexpinn')
INPUT_BUNDLE = DRIVE_ROOT / 'inputs/apexpinn_t13_input_v1.zip'
BUNDLE_SHA256 = 'b787682f95e30b303bfe7a41a2739dd40051a200724f079ffcab84a7d5618a03'
OVERLAY_BUNDLE = DRIVE_ROOT / 'inputs/apexpinn_m1_diagnostic_v1.zip'
OVERLAY_SHA256 = 'f81ab87448a292b5c2b9a6444083a4856e9ffda0259d281f12ba7cbbdb8ec506'
SOURCE_RUN = DRIVE_ROOT / 'runs/t13_m1_f1_mid_patch01'
DIAGNOSTIC_ID = 'm1_offsets_f1_mid_v1'
OUTPUT = DRIVE_ROOT / 'diagnostics' / DIAGNOSTIC_ID
if not DIAGNOSTIC_ID or any(c not in 'abcdefghijklmnopqrstuvwxyz0123456789_-' for c in DIAGNOSTIC_ID):
    raise ValueError('Use a simple, unique diagnostic ID.')
if not (SOURCE_RUN / 'jobs/M1_F1_mid_main/job.json').is_file():
    raise FileNotFoundError('The completed M1 run is required on Drive.')


## 3. Verify and restore the separate code workspace

Reuse the original approved input ZIP parts and verify the diagnostic update independently.
Completed run files stay on Drive and are read only. New reports go under `apexpinn/diagnostics`.


### Restore the input from Drive upload parts

The connector limits each file to 100 MiB. These two parts contain the unchanged original ZIP. This cell checks every part and the full archive SHA, then writes the ZIP on the remote VM. It performs file I/O only. No model is fitted. The original notebook and bundle are preserved.

In [ ]:
import hashlib
import json
import re
import tempfile

def assemble_input_bundle(input_bundle, expected_sha, destination_directory):
    manifest = json.loads(Path(str(input_bundle) + '.parts.json').read_text())
    if manifest['archive_sha256'] != expected_sha:
        raise ValueError('Part manifest differs from the approved input receipt.')
    checksum = hashlib.sha256()
    with tempfile.NamedTemporaryFile(
        mode='wb', prefix='apexpinn_input_', suffix='.zip',
        dir=destination_directory, delete=False
    ) as output:
        assembled = Path(output.name)
        for part in manifest['parts']:
            if not re.fullmatch(re.escape(input_bundle.name) + r'\.part[0-9]{2}', part['name']):
                raise ValueError('Unsafe input part name.')
            part_hash = hashlib.sha256()
            count = 0
            with (input_bundle.parent / part['name']).open('rb') as stream:
                for block in iter(lambda: stream.read(1 << 20), b''):
                    output.write(block)
                    checksum.update(block)
                    part_hash.update(block)
                    count += len(block)
            if count != part['bytes'] or part_hash.hexdigest() != part['sha256']:
                raise ValueError('Input part hash or size differs.')
    if checksum.hexdigest() != expected_sha:
        raise ValueError('Assembled archive differs from the approved input SHA-256.')
    return assembled

if not INPUT_BUNDLE.exists():
    INPUT_BUNDLE = assemble_input_bundle(INPUT_BUNDLE, BUNDLE_SHA256, Path('/content'))
    print('Reassembled and hash-verified input ZIP on the remote VM:', INPUT_BUNDLE)


In [ ]:
import hashlib
import json

def file_sha(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if file_sha(INPUT_BUNDLE) != BUNDLE_SHA256:
    raise ValueError('Input archive hash mismatch.')
sys.path.insert(0, str(INPUT_BUNDLE) + '/src')
from apexpinn.cloud.transfer import unpack
import importlib.util
from apexpinn.cloud.transfer import unpack

if file_sha(OVERLAY_BUNDLE) != OVERLAY_SHA256:
    raise ValueError('Update archive hash mismatch')
OVERLAY_ROOT = Path('/content') / ('apexpinn_update_' + OVERLAY_SHA256[:12])
if not OVERLAY_ROOT.exists():
    unpack(OVERLAY_BUNDLE, OVERLAY_ROOT, expected_sha256=OVERLAY_SHA256, kind='input')
else:
    import zipfile
    with zipfile.ZipFile(OVERLAY_BUNDLE) as update:
        expected_overlay = json.loads(update.read('bundle_manifest.json'))
    if json.loads((OVERLAY_ROOT / 'bundle_manifest.json').read_text()) != expected_overlay:
        raise ValueError('Update manifest changed')
    for name, item in expected_overlay['files'].items():
        if file_sha(OVERLAY_ROOT / name) != item['sha256']:
            raise ValueError('Update workspace changed')
spec = importlib.util.spec_from_file_location(
    'apexpinn_handoff', OVERLAY_ROOT / 'src/apexpinn/cloud/handoff.py')
handoff = importlib.util.module_from_spec(spec)
spec.loader.exec_module(handoff)
WORKSPACE = Path('/content') / ('apexpinn_' + BUNDLE_SHA256[:8] + '_' + OVERLAY_SHA256[:12])
handoff.restore_overlay(INPUT_BUNDLE, OVERLAY_ROOT, WORKSPACE, base_sha256=BUNDLE_SHA256)
print('Verified updated workspace:', WORKSPACE)


## 4. Restore the locked Python 3.12 GPU environment

This verifies GPU and float64, without optimizer updates. The diagnostic also checks original
package versions, source bytes, Gold identity, model/checkpoint hashes and training offsets.


In [ ]:
import shutil
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
UV = shutil.which('uv')
subprocess.run(
    [UV, 'sync', '--frozen', '--extra', 'cuda', '--no-dev', '--python', '3.12'],
    cwd=WORKSPACE, check=True)
PYTHON = WORKSPACE / '.venv/bin/python'
subprocess.run([str(PYTHON), '-c',
    'import jax; jax.config.update("jax_enable_x64", True); '
    'from apexpinn.train.cloud_checkpoint import require_colab_gpu; '
    'require_colab_gpu(); print(jax.devices(), jax.config.x64_enabled)'], cwd=WORKSPACE, check=True)


## 5. Run the saved-model diagnostic

This runs model inference on Colab. It deserializes optimizer state to recover the saved offsets,
but never computes gradients or updates the optimizer. Ten completed fits are inspected.
An existing output directory is refused. A partial diagnostic needs a new ID, not a training rerun.
Full process output and exit status persist under `apexpinn/logs/DIAGNOSTIC_ID`.


In [ ]:
LOG_ROOT = DRIVE_ROOT / 'logs' / DIAGNOSTIC_ID
command = [str(PYTHON), 'scripts/diagnose_m1.py', '--run', str(SOURCE_RUN), '--out', str(OUTPUT)]
subprocess.run([str(PYTHON), '-m', 'apexpinn.cloud.process', '--cwd', str(WORKSPACE),
                '--logs', str(LOG_ROOT), '--', *command], check=True)


## 6. Read the separate report

A large offset contribution locates training pace error. Remaining profiled error locates shape
misfit. Neither establishes that a changed method will improve held-out forecasts. Original M1
scores remain unchanged. No output has been computed in this prepared notebook.


In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown((OUTPUT / 'diagnostic.md').read_text()))
display(Image(filename=str(OUTPUT / 'offset_errors.png')))
display(Image(filename=str(OUTPUT / 'example_curve.png')))


## 7. Export diagnostics for review
Export only the new diagnostic folder.


In [ ]:
if not (OUTPUT / 'diagnostic.json').is_file():
    raise RuntimeError('No completed diagnostic receipt; inspect persistent logs.')
shutil.copytree(LOG_ROOT, OUTPUT / 'execution', dirs_exist_ok=True)
EXPORT = DRIVE_ROOT / 'exports' / (DIAGNOSTIC_ID + '_results_01.zip')
EXPORT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([str(PYTHON), '-m', 'apexpinn.cloud.transfer', 'pack-results',
                '--root', str(OUTPUT), '--out', str(EXPORT)], cwd=WORKSPACE, check=True)
print('Result archive:', EXPORT)
print('Receipt:', str(EXPORT) + '.receipt.json')
print('SHA-256:', file_sha(EXPORT))
